# Random Forest Model Training

This notebook contains the code to train a static Random Forest classifier on the preprocessed CICEVSE2024 dataset.

In [ ]:
import os
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

PROCESSED_DATA_DIR = "../../../data/processed"
MODEL_SAVE_DIR = "../../../saved_models"
PREDICTIONS_DIR = "../../../predictions"

def load_data(data_dir: str):
    """Loads train, validation, and test sets."""
    print(f"Loading preprocessed data from '{data_dir}'...")
    X_train = pd.read_csv(os.path.join(data_dir, "X_train.csv"))
    y_train = pd.read_csv(os.path.join(data_dir, "y_train.csv"))
    
    X_val = pd.read_csv(os.path.join(data_dir, "X_val.csv"))
    y_val = pd.read_csv(os.path.join(data_dir, "y_val.csv"))
    
    X_test = pd.read_csv(os.path.join(data_dir, "X_test.csv"))
    y_test = pd.read_csv(os.path.join(data_dir, "y_test.csv"))
    
    print(f"Loaded successfully. X_train shape: {X_train.shape}")
    return X_train, y_train["Label_Binary"].values.ravel(), X_val, y_val["Label_Binary"].values.ravel(), X_test, y_test["Label_Binary"].values.ravel()

# 1. Load Data
X_train, y_train, X_val, y_val, X_test, y_test = load_data(PROCESSED_DATA_DIR)

# 2. Train Model
print("Initializing RandomForestClassifier...")
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

print("Training the model...")
rf_model.fit(X_train, y_train)
print("Training completed.")

# 3. Evaluate Model
print("Evaluating model on the test set...")
y_pred = rf_model.predict(X_test)

print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred, zero_division=0))

# 4. Save Model & Predictions
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
joblib.dump(rf_model, os.path.join(MODEL_SAVE_DIR, "random_forest.pkl"))

os.makedirs(PREDICTIONS_DIR, exist_ok=True)
pd.DataFrame({
    "True_Label": y_test,
    "Predicted_Label": y_pred
}).to_csv(os.path.join(PREDICTIONS_DIR, "rf_predictions.csv"), index=False)
print("Saved model and predictions successfully!")